In [22]:
from dotenv import load_dotenv
load_dotenv()

True

### imports

In [23]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

### laod pdf

In [24]:
loader = PyPDFLoader("../data/data_science_syllabus.pdf")
docs  = loader.load()


split document

In [25]:
splitter  = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap = 200
)
splitted_data = splitter.split_documents(docs)

### embeddingd

In [26]:
embeddings = GoogleGenerativeAIEmbeddings(
     model="gemini-embedding-2-preview"
)

### remove empty chunk

In [27]:
splitted_data = [
    doc for doc in splitted_data
    if doc.page_content and doc.page_content.strip() != ""
]


In [44]:
from dotenv import load_dotenv
import os
import chromadb
import shutil
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv(dotenv_path="../.env")

# Embeddings - model change kiya
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

shutil.rmtree("./new_chroma_db", ignore_errors=True)

client = chromadb.PersistentClient(path="./new_chroma_db")

vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings,
    client=client,
    collection_name="my_collection"
)

query = "Machine Learnings and Data Science Content"
data = vector_store.similarity_search(query=query)
print(data)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


GoogleGenerativeAIError: Error embedding content (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/embedding-001 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

### vector store

In [ ]:
context = ""
for doc in data:
    context += doc.page_content + "\n"


NameError: name 'data' is not defined

### model

In [ ]:
llm = ChatGoogleGenerativeAI(model="models/gemini-flash-latest")

### chain - context_generate | prompt | llm | strparser

### similar serach


In [ ]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }

In [ ]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question}
""")

In [ ]:
rag_chain = get_context | prompt | llm

In [ ]:
res = rag_chain.invoke("what are var in javascript? ")

NameError: name 'vector_store' is not defined

In [ ]:

print(res.content)